In [ ]:
!pip install -q transformers datasets torch scikit-learn matplotlib tqdm

# FinBert - Pre Train model for Finance

  1. Finbert-ESG
  
    - Four classifications: Ambiental, Social, Gobernanza o Ninguna.

In [ ]:
# Simple one-shot demo: classify short financial paragraphs as E / S / G / None
# Run in Colab or local Python (requires: pip install transformers torch)
import torch
from transformers import pipeline

# Use GPU if available
device = 0 if torch.cuda.is_available() else -1

# Load FinBERT-ESG sequence classification pipeline (no fine-tuning required)
clf = pipeline(
    "text-classification",
    model="yiyanghkust/finbert-esg",
    tokenizer="yiyanghkust/finbert-esg",
    device=device,
    return_all_scores=False  # set True if you want scores for all labels
)

In [ ]:
# Very short example texts (one-shot trial)
examples = [
    "The company committed to net zero and cut its scope 1 emissions by 40% this year.",
    "Several factory workers reported unsafe working conditions and lack of protective gear.",
    "Board members are related-party and disclosure practices are weak.",
    "Revenue grew 12% this quarter driven by new product sales."
]

# Run inference and print results
results = clf(examples, truncation=True)
for text, res in zip(examples, results):
    label = res['label']
    score = res.get('score', None)
    print(f"Text: {text}\n -> Predicted: {label}  (score: {score:.3f})\n")


  2. Finbert-ESG 9 categories
  
    - Nine classifications: Climate Change, Natural Capital, Pollution & Waste, Human Capital, Product Liability, Community Relations, Corporate Governance, Business Ethics & Values, or Non-ESG

In [ ]:
# Load FinBERT-ESG sequence classification pipeline (no fine-tuning required)
clf = pipeline(
    "text-classification",
    model="yiyanghkust/finbert-esg-9-categories",
    tokenizer="yiyanghkust/finbert-esg-9-categories",
    device=device,
    return_all_scores=False  # set True if you want scores for all labels
)

In [ ]:
# Very short example texts (one-shot trial)
examples = [
    "The company committed to net zero and cut its scope 1 emissions by 40% this year.",
    "Several factory workers reported unsafe working conditions and lack of protective gear.",
    "Board members are related-party and disclosure practices are weak.",
    "Revenue grew 12% this quarter driven by new product sales."
]

# Run inference and print results
results = clf(examples, truncation=True)
for text, res in zip(examples, results):
    label = res['label']
    score = res.get('score', None)
    print(f"Text: {text}\n -> Predicted: {label}  (score: {score:.3f})\n")

# Image Models

  1. Vision Transformer pretrained on ImageNet (works out-of-the-box for image classification) - google/vit-base-patch16-224

  - This model looks at an image by cutting it into small pieces (“patches”) and then figures out what it shows — for example, a cat, a car, or a tree. It works kind of like how people read, piece by piece, and then understand the full picture.

In [ ]:
!pip install -q pillow

In [ ]:
# Colab cell: ViT image classification (PyTorch)
from PIL import Image
import requests
from io import BytesIO

device = 0 if torch.cuda.is_available() else -1
clf = pipeline("image-classification", model="google/vit-base-patch16-224", device=device)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms

# Load CIFAR-10
cifar10 = datasets.CIFAR10(root="./data", train=False, download=True, transform=transforms.ToTensor())


In [ ]:
# Pick one sample image and label
img, label = cifar10[0]

# Show the image
plt.imshow(np.transpose(img.numpy(), (1, 2, 0)))
plt.axis("off")
plt.title(f"Ground truth label: {cifar10.classes[label]}")
plt.show()

# Convert tensor -> PIL Image for Hugging Face pipeline
from torchvision.transforms.functional import to_pil_image
img_pil = to_pil_image(img)

# Run classification
results = clf(img_pil, top_k=5)
print("Top predictions (ViT):")
for r in results:
    print(f"{r['label']}: {r['score']:.4f}")

In [ ]:
# Pick one sample image and label
img2, label2 = cifar10[1]

# Show the image
plt.imshow(np.transpose(img2.numpy(), (1, 2, 0)))
plt.axis("off")
plt.title(f"Ground truth label: {cifar10.classes[label2]}")
plt.show()

# Convert tensor -> PIL Image for Hugging Face pipeline
from torchvision.transforms.functional import to_pil_image
img_pil2 = to_pil_image(img2)

# Run classification
results2 = clf(img_pil2, top_k=5)
print("Top predictions (ViT):")
for r in results2:
    print(f"{r['label']}: {r['score']:.4f}")

  2. Object detection pretrained -DETR panoptic segmentation (detects bounding boxes + labels on COCO classes) - facebook/detr-resnet-50

  - This model not only sees what is in a picture, but also where each object is located. It can say “there’s a dog on the left” and “a ball on the right,” making it useful for identifying multiple things at once.

In [ ]:
!pip install -q torchvision

In [ ]:
# Import necessary libraries
import torch
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from transformers import AutoFeatureExtractor, DetrForObjectDetection
from google.colab import files
import requests # Added for a default image if no upload

# --- 1. Load Image ---
# Upload an image from your computer or use a default one
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
img = Image.open(image_path).convert("RGB")

# --- 2. Load Model ---
# We use DetrForObjectDetection as it's more direct for this task
feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")

In [ ]:
# --- 3. Process Image and Get Predictions ---
# Prepare image for the model
inputs = feature_extractor(images=img, return_tensors="pt")

# Run model
with torch.no_grad():
    outputs = model(**inputs)

# Post-process the output to get bounding boxes, labels, and scores
# We use a threshold to filter out low-confidence detections
target_sizes = torch.tensor([img.size[::-1]])
results = feature_extractor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.5)[0]

# --- 4. Prepare Images for Display ---

# Create copies of the original image to draw on
img_boxes = img.copy()
img_boxes_labels = img.copy()

# Initialize drawing contexts
draw_boxes = ImageDraw.Draw(img_boxes)
draw_boxes_labels = ImageDraw.Draw(img_boxes_labels)

# Loop through detected objects
for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    box = [round(i, 2) for i in box.tolist()]
    label_text = model.config.id2label[label.item()]

    # Draw bounding box on the first copy
    draw_boxes.rectangle(box, outline="red", width=3)

    # Draw bounding box and label on the second copy
    draw_boxes_labels.rectangle(box, outline="red", width=3)
    draw_boxes_labels.text((box[0], box[1]), f"{label_text}: {score:.2f}", fill="red")

# --- 5. Display the Three Images ---
plt.figure(figsize=(21, 7))

# Plot 1: Original Image
plt.subplot(1, 3, 1)
plt.imshow(img)
plt.axis("off")
plt.title("1. Original Image")

# Plot 2: Image with Bounding Boxes
plt.subplot(1, 3, 2)
plt.imshow(img_boxes)
plt.axis("off")
plt.title("2. Image with Bounding Boxes")

# Plot 3: Image with Bounding Boxes and Labels
plt.subplot(1, 3, 3)
plt.imshow(img_boxes_labels)
plt.axis("off")
plt.title("3. Image with Boxes, Labels & Scores")

plt.tight_layout()
plt.show()

# Optional: Print detected objects to console
print("\nDetected objects:")
for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    print(f"• {model.config.id2label[label.item()]} (score: {score:.2f})")

In [ ]:
# --- 1. Load Image ---
# Upload an image from your computer or use a default one
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
img = Image.open(image_path).convert("RGB")

# --- 3. Process Image and Get Predictions ---
# Prepare image for the model
inputs = feature_extractor(images=img, return_tensors="pt")

# Run model
with torch.no_grad():
    outputs = model(**inputs)

# Post-process the output to get bounding boxes, labels, and scores
# We use a threshold to filter out low-confidence detections
target_sizes = torch.tensor([img.size[::-1]])
results = feature_extractor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.5)[0]

# --- 4. Prepare Images for Display ---

# Create copies of the original image to draw on
img_boxes = img.copy()
img_boxes_labels = img.copy()

# Initialize drawing contexts
draw_boxes = ImageDraw.Draw(img_boxes)
draw_boxes_labels = ImageDraw.Draw(img_boxes_labels)

# Loop through detected objects
for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    box = [round(i, 2) for i in box.tolist()]
    label_text = model.config.id2label[label.item()]

    # Draw bounding box on the first copy
    draw_boxes.rectangle(box, outline="red", width=3)

    # Draw bounding box and label on the second copy
    draw_boxes_labels.rectangle(box, outline="red", width=3)
    draw_boxes_labels.text((box[0], box[1]), f"{label_text}: {score:.2f}", fill="red")

# --- 5. Display the Three Images ---
plt.figure(figsize=(21, 7))

# Plot 1: Original Image
plt.subplot(1, 3, 1)
plt.imshow(img)
plt.axis("off")
plt.title("1. Original Image")

# Plot 2: Image with Bounding Boxes
plt.subplot(1, 3, 2)
plt.imshow(img_boxes)
plt.axis("off")
plt.title("2. Image with Bounding Boxes")

# Plot 3: Image with Bounding Boxes and Labels
plt.subplot(1, 3, 3)
plt.imshow(img_boxes_labels)
plt.axis("off")
plt.title("3. Image with Boxes, Labels & Scores")

plt.tight_layout()
plt.show()

# Optional: Print detected objects to console
print("\nDetected objects:")
for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    print(f"• {model.config.id2label[label.item()]} (score: {score:.2f})")



  3. Image captioning (vision→text) — Salesforce/blip2-opt-2.7b (BLIP-2)

  - This is a smart model that can describe images with words or answer questions about them. For example, if you show it a photo of a cat on a sofa and ask “What is the cat doing?”, it can answer “The cat is sitting on the sofa.”

- This model is a mini version of 8bits due to the high memory neccesary for the whole model

In [ ]:
# Install required libraries in Colab
!pip install -q bitsandbytes accelerate

In [ ]:
import torch
from PIL import Image
import requests
from transformers import AutoProcessor, Blip2ForConditionalGeneration
import matplotlib.pyplot as plt

# --- 1. Setup ---
# Set device to GPU if available, otherwise CPU
# NOTE: For 8-bit, device_map="auto" handles this better than manually setting the device.
print("Checking for CUDA availability...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- KEY CHANGE: Load the model in 8-bit precision ---
# This significantly reduces memory usage.
# We use device_map="auto" to let accelerate handle placing the model on the GPU.
print("Loading model in 8-bit precision...")
processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    load_in_8bit=True,
    device_map="auto"
)
print("Model loaded.")

# Load a sample image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# --- 2. Example 1: Image Captioning ---
print("\n--- Example 1: Generating a Description ---")

# We don't need to specify torch_dtype for inputs with a quantized model
inputs = processor(images=image, return_tensors="pt").to(device)

generated_ids = model.generate(**inputs, max_new_tokens=50)
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
print(f"Generated Caption: {generated_text}")

# --- 3. Example 2: Visual Question Answering (VQA) ---
print("\n--- Example 2: Answering Questions About the Image ---")

def ask_question(question):
    """A helper function to ask a question and print the answer."""
    prompt = f"Question: {question} Answer:"

    # Process both image and text
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)

    generated_ids = model.generate(**inputs, max_new_tokens=20)
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    print(f"Q: {question}")
    print(f"A: {generated_text}")
    print("-" * 30)

ask_question("How many cats are in the picture?")
ask_question("What are the cats doing?")

- Model Smaller with 16-bit

In [ ]:
import torch
from PIL import Image
import requests
from transformers import AutoProcessor, Blip2ForConditionalGeneration
import matplotlib.pyplot as plt

# --- 1. Setup ---
# Set device to GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the processor and model from Hugging Face
# Using float16 for less memory usage
processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16
).to(device)

# Load a sample image from an open-source dataset (COCO)
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# Display the image we're working with
print("--- Displaying Source Image ---")
plt.imshow(image)
plt.title("Source Image")
plt.axis("off")
plt.show()


In [ ]:
# --- 2. Example 1: Image Captioning ---

print("\n--- Example 1: Generating a Description ---")

# Prepare the image for the model using the processor
# No text prompt is needed for basic captioning
inputs = processor(images=image, return_tensors="pt").to(device, torch.float16)

# Generate a caption
generated_ids = model.generate(**inputs, max_new_tokens=50)

# Decode the generated IDs to text
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

# Print the result
print(f"Generated Caption: {generated_text}")

In [ ]:
# --- 3. Example 2: Visual Question Answering (VQA) ---

print("\n--- Example 2: Answering Questions About the Image ---")

def ask_question(question):
    """A helper function to ask a question and print the answer."""
    # Format the prompt with the question
    prompt = f"Question: {question} Answer:"

    # Process both the image and the text prompt
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device, torch.float16)

    # Generate an answer
    generated_ids = model.generate(**inputs, max_new_tokens=20)

    # Decode and print the result
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    print(f"Q: {question}")
    print(f"A: {generated_text}")
    print("-" * 30)

# Ask a few different questions about the image
ask_question("How many cats are in the picture?")
ask_question("What color is the couch?")
ask_question("What are the cats doing?")

3.1 Ligther version can be git-base-coco

In [ ]:
import torch
from PIL import Image
import requests
from transformers import AutoProcessor, AutoModelForCausalLM
import matplotlib.pyplot as plt

In [ ]:
# --- 1. Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = "microsoft/git-base-coco"
print(f"Loading smaller model: {model_name}")

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
print("Model loaded.")

In [ ]:
# Load the same sample image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# --- 2. Example 1: Image Captioning  ---
print("\n--- Example 1: Generating a Description ---")

# For captioning, we only process the image
caption_inputs = processor(images=image, return_tensors="pt").to(device)
pixel_values = caption_inputs.pixel_values

generated_ids = model.generate(pixel_values=pixel_values, max_length=50)
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(f"Generated Caption: {generated_text}")

In [ ]:
# --- 3. Example 2: Visual Question Answering (VQA) ---
print("\n--- Example 2: Answering Questions About the Image ---")

def ask_question_git(image, question):
    """
    A helper function for the GIT model that uses the correct
    combined-modality processing and isolates the answer.
    """
    # BEST LOGIC: Process the image and text question TOGETHER.
    # This is the crucial step that ensures the model understands
    # it needs to answer a question about this specific image.
    inputs = processor(images=image, text=question, return_tensors="pt").to(device)

    # Get the length of the question tokens
    prompt_length = inputs.input_ids.shape[1]

    # Generate the answer
    generated_ids = model.generate(**inputs, max_new_tokens=50)

    # Slice the output to remove the question and get only the answer
    answer_ids = generated_ids[:, prompt_length:]

    # Decode the answer
    answer = processor.batch_decode(answer_ids, skip_special_tokens=True)[0].strip()

    print(f"Q: {question}")
    print(f"A: {answer}")
    print("-" * 30)

# Use the corrected helper function to ask questions
ask_question_git(image, "how many cats are in the picture?")
ask_question_git(image, "what are the cats doing?")
ask_question_git(image, "what color is the remote control?")

  4. Semantic segmentation — nvidia/segformer-b0-finetuned-ade-512-512

  - Its purpose is to assign a class label to every pixel in an image, making it useful for applications such as autonomous driving, robotics navigation, urban scene understanding, and image editing.

In [ ]:
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import pipeline
from google.colab import files # Import the files module for uploading

# --- 1. Setup Model ---

# Set device to GPU if available for faster processing
device = 0 if torch.cuda.is_available() else -1

# Load the segmentation pipeline with the specified SegFormer model
print("Loading the segmentation model (this may take a moment)...")
segmenter = pipeline("image-segmentation", model="nvidia/segformer-b0-finetuned-ade-512-512", device=device)
print("Model loaded successfully.")

# --- 2. Upload and Load Your Image ---
print("\nPlease upload an image file.")
uploaded = files.upload()

# Check if a file was uploaded
if not uploaded:
    print("\nNo file uploaded. Please run the cell again to upload an image.")
else:
    # Get the filename of the uploaded file
    image_path = list(uploaded.keys())[0]
    # Open the image
    image = Image.open(image_path).convert("RGB")
    print(f"\nSuccessfully loaded image: {image_path}")


    # --- 3. Run Inference ---

    # Get the segmentation masks from the model
    print("Running segmentation on the image...")
    masks = segmenter(image)
    print("Segmentation complete.")


    # --- 4. Visualize the Results ---

    # Create a color map for visualization
    unique_labels = {mask['label'] for mask in masks}
    color_map = {label: list(np.random.choice(range(256), size=3)) for label in unique_labels}

    # Create an empty RGB image to draw the segmentation map
    segmentation_map = np.zeros((image.height, image.width, 3), dtype=np.uint8)

    # Overlay each mask with its corresponding color
    for mask_info in masks:
        label = mask_info['label']
        mask = np.array(mask_info['mask'])
        color = color_map[label]
        segmentation_map[mask > 0] = color

    # Create a blended image for a nice overlay effect
    alpha = 0.6  # Transparency of the overlay
    blended_image = Image.fromarray(
        (alpha * np.array(image) + (1 - alpha) * segmentation_map).astype(np.uint8)
    )

    # --- 5. Display the Images and a Legend ---

    fig, axs = plt.subplots(1, 2, figsize=(18, 9))

    axs[0].imshow(image)
    axs[0].axis('off')
    axs[0].set_title("Original Uploaded Image")

    axs[1].imshow(blended_image)
    axs[1].axis('off')
    axs[1].set_title("Semantic Segmentation Overlay")

    patches = [mpatches.Patch(color=np.array(color)/255, label=label) for label, color in color_map.items()]
    plt.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

    plt.tight_layout()
    plt.show()


In [ ]:
# --- 2. Upload and Load Your Image ---
print("\nPlease upload an image file.")
uploaded = files.upload()

# Check if a file was uploaded
if not uploaded:
    print("\nNo file uploaded. Please run the cell again to upload an image.")
else:
    # Get the filename of the uploaded file
    image_path = list(uploaded.keys())[0]
    # Open the image
    image = Image.open(image_path).convert("RGB")
    print(f"\nSuccessfully loaded image: {image_path}")


    # --- 3. Run Inference ---

    # Get the segmentation masks from the model
    print("Running segmentation on the image...")
    masks = segmenter(image)
    print("Segmentation complete.")


    # --- 4. Visualize the Results ---

    # Create a color map for visualization
    unique_labels = {mask['label'] for mask in masks}
    color_map = {label: list(np.random.choice(range(256), size=3)) for label in unique_labels}

    # Create an empty RGB image to draw the segmentation map
    segmentation_map = np.zeros((image.height, image.width, 3), dtype=np.uint8)

    # Overlay each mask with its corresponding color
    for mask_info in masks:
        label = mask_info['label']
        mask = np.array(mask_info['mask'])
        color = color_map[label]
        segmentation_map[mask > 0] = color

    # Create a blended image for a nice overlay effect
    alpha = 0.6  # Transparency of the overlay
    blended_image = Image.fromarray(
        (alpha * np.array(image) + (1 - alpha) * segmentation_map).astype(np.uint8)
    )

    # --- 5. Display the Images and a Legend ---

    fig, axs = plt.subplots(1, 2, figsize=(18, 9))

    axs[0].imshow(image)
    axs[0].axis('off')
    axs[0].set_title("Original Uploaded Image")

    axs[1].imshow(blended_image)
    axs[1].axis('off')
    axs[1].set_title("Semantic Segmentation Overlay")

    patches = [mpatches.Patch(color=np.array(color)/255, label=label) for label, color in color_map.items()]
    plt.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

    plt.tight_layout()
    plt.show()

In [ ]:
# Check if a file was uploaded
if not uploaded:
    print("\nNo file uploaded. Please run the cell again to upload an image.")
else:
    # Get the filename of the uploaded file
    image_path = list(uploaded.keys())[0]
    # Open the image
    image = Image.open(image_path).convert("RGB")
    print(f"\nSuccessfully loaded image: {image_path}")


    # --- 3. Run Inference ---

    # Get the segmentation masks from the model
    print("Running segmentation on the image...")
    masks = segmenter(image)
    print("Segmentation complete.")


    # --- 4. Visualize the Results ---

    # Create a color map for visualization
    unique_labels = {mask['label'] for mask in masks}
    color_map = {label: list(np.random.choice(range(256), size=3)) for label in unique_labels}

    # Create an empty RGB image to draw the segmentation map
    segmentation_map = np.zeros((image.height, image.width, 3), dtype=np.uint8)

    # Overlay each mask with its corresponding color
    for mask_info in masks:
        label = mask_info['label']
        mask = np.array(mask_info['mask'])
        color = color_map[label]
        segmentation_map[mask > 0] = color

    # Create a blended image for a nice overlay effect
    alpha = 0.6  # Transparency of the overlay
    blended_image = Image.fromarray(
        (alpha * np.array(image) + (1 - alpha) * segmentation_map).astype(np.uint8)
    )

    # --- 5. Display the Images and a Legend ---

    fig, axs = plt.subplots(1, 2, figsize=(18, 9))

    axs[0].imshow(image)
    axs[0].axis('off')
    axs[0].set_title("Original Uploaded Image")

    axs[1].imshow(blended_image)
    axs[1].axis('off')
    axs[1].set_title("Semantic Segmentation Overlay")

    patches = [mpatches.Patch(color=np.array(color)/255, label=label) for label, color in color_map.items()]
    plt.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

    plt.tight_layout()
    plt.show()